In [ ]:
import pandas as pd

df = pd.read_parquet(r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet\A123\METABatt_A123_APR18650M1B_007.parquet")

In [ ]:
df

In [ ]:
from minio import Minio
import urllib3


working_path = r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet"
type_cell = "VTC"


minio_endpoint = "optimusprime.isea.rwth-aachen.de:9000"
access_key= "8ms0O8n4gwMia5BpDrYq"
secret_key= "WxDnJMQmUrW8RScViRf0CCTBDDIlZaoIANQgTFWl"
bucket_name= "zho"
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


minio_client = Minio(
    minio_endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=True,
    cert_check=False,
)


In [ ]:
from importlib import reload
from output import add_export_file
reload(add_export_file)
from output.add_information_METABATT import add_additional_information
import os
import glob
import pandas as pd

List_Cell = glob.glob1(working_path+f'\{type_cell}', '*.parquet')


Project_Schedule = pd.DataFrame()
df_results_cap = pd.DataFrame()
df_results_pulse = pd.DataFrame()
processed_count = 0

exception_dict = {}

for cell in List_Cell:
        try:
                savepath_df_GOLD = os.path.join(
                working_path, type_cell, cell)

                df_results_cap,df_results_pulse, exception_dict_no_cap, success = add_export_file.add_export_file(savepath_df_GOLD, cell, df_results_cap,df_results_pulse, exception_dict)
                #if success:
                        #Project_Schedule = add_test_schedule.preparing_schedule_overview(df_after_filter, Project_Schedule, cell)
        except Exception as e:
                print(f"Warning: there was an error: {cell}: {type(e).__name__}: {e}")
                continue



#add_additional_information(df_results_cap)
add_additional_information(df_results_pulse)

#df_results_cap.to_csv('Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten/capacity_results_'+type_cell+'.csv', index=False)
df_results_pulse.to_csv('Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten/pulse_results_'+type_cell+'.csv', index=False)



In [ ]:
List_Cell

In [ ]:
import s3fs

# Create S3 filesystem
fs = s3fs.S3FileSystem(
    key=access_key,
    secret=secret_key,
    endpoint_url="https://optimusprime.isea.rwth-aachen.de:9000",
    use_ssl=False,
    client_kwargs={'verify': False}
)

df = pd.read_parquet(f"s3://zho/Metabatt/GOLD/{type_cell}/{cell}", filesystem=fs)

In [ ]:
df['target'].dropna().unique()

In [ ]:
df['Capacity_py'].unique()


In [ ]:
df_results_cap

In [ ]:
pd.read_parquet(r"D:\Data\METABatt\VTC\BRONZE_CU\METABatt_Sony_Murata_18650VTC6_109.parquet").groupby('Ahjo_Test_ID').head(1)

In [ ]:
add_additional_information(df_results_cap)

In [ ]:
df_results_cap

In [ ]:
df_results_cap_old = pd.read_csv(r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Auswertung\archieve\capacity_results_"+type_cell+".csv")

In [ ]:
columns_to_fill = ["DOD","SOC","C_Rate","Temperature"]
# Get first value for each Name from df1
df1_first = df_results_cap_old.groupby('Name')[columns_to_fill].first()

# Update df2's missing values
for col in columns_to_fill:
    df_results_cap[col] = df_results_cap[col].fillna(df_results_cap['Name'].map(df1_first[col]))

In [ ]:
df_results_cap.reset_index(drop=True, inplace=True)

In [ ]:
df_results_cap

In [ ]:
df_results_cap.to_csv('Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten/capacity_results_'+type_cell+'.csv', index=False)

In [ ]:
export_kapa = df_results_cap.loc[df_results_cap.groupby("Name")["Capacity_py"].idxmin()]

In [ ]:
export_kapa.to_csv('Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten/capacity_results_'+type_cell+ '_latest_value.csv', index=False)

In [ ]:
export_kapa

In [ ]:
export_kapa